
# Ευφυές Σύστημα Κυβερνοασφάλειας  
## Ταξινόμηση Δικτυακής Κίνησης Tor / Non-Tor με Random Forest

**Στόχος:** Ανάπτυξη ευφυούς συστήματος ταξινόμησης δικτυακής κίνησης που προέρχεται ή όχι από το δίκτυο Tor, με βάση το σύνολο δεδομένων **DarkNet.csv**.

**Παραδοτέα που καλύπτει το notebook:**  
- Ζητούμενο (2): Colab σημειωματάριο (.ipynb)  
- Ζητούμενο (3): Τεκμηρίωση κώδικα με σχόλια `#...`


In [ ]:

# (Προαιρετικό) Εγκατάσταση απαραίτητων βιβλιοθηκών, αν το περιβάλλον δεν τις έχει ήδη.
# Συνήθως στο Google Colab υπάρχουν ήδη.
# !pip install pandas scikit-learn matplotlib


In [ ]:

# Εισαγωγή βασικών βιβλιοθηκών
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:

# Φόρτωση του συνόλου δεδομένων DarkNet.csv
# Στο Colab: (1) ανεβάζετε το αρχείο DarkNet.csv ή (2) το τοποθετείτε στο ίδιο directory.
df = pd.read_csv('DarkNet.csv')

# Εμφάνιση των πρώτων γραμμών
df.head()


In [ ]:

# Βασικές πληροφορίες για το dataset (τύποι στηλών, κενές τιμές κ.λπ.)
df.info()


In [ ]:

# Στατιστική περιγραφή αριθμητικών χαρακτηριστικών
df.describe()


In [ ]:

# Έλεγχος κατανομής κλάσεων (Tor / Non-Tor) με απλό γράφημα
class_counts = df['Label-1'].value_counts()

ax = class_counts.plot(kind='bar')
ax.set_title('Κατανομή κλάσεων Tor / Non-Tor (Label-1)')
ax.set_xlabel('Κλάση')
ax.set_ylabel('Πλήθος εγγραφών')
plt.show()

class_counts


In [ ]:

# Επιλογή ετικέτας για το πρόβλημα ταξινόμησης
# Label-1: Tor / Non-Tor (στόχος μας)
y = df['Label-1']

# Επιλογή χαρακτηριστικών (features)
# Αφαιρούμε τις ετικέτες Label-1 και Label-2 από τα εισερχόμενα χαρακτηριστικά
X = df.drop(columns=['Label-1', 'Label-2'])


In [ ]:

# Διαχωρισμός σε train/test σύνολα
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:

# Εκπαίδευση μοντέλου Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,   # περισσότερα δέντρα για σταθερότερη απόδοση
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)


In [ ]:

# Πρόβλεψη στο test set
y_pred = rf_model.predict(X_test)


In [ ]:

# Αξιολόγηση μοντέλου (Accuracy, Precision/Recall/F1)
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


In [ ]:

# Confusion Matrix (οπτικοποίηση) με sklearn
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_model.classes_)
disp.plot()
plt.title('Confusion Matrix')
plt.show()



## Feature Importance (Ερμηνευσιμότητα)

Η **σημαντικότητα χαρακτηριστικών (feature importance)** δείχνει ποια χαρακτηριστικά συμβάλλουν περισσότερο στις αποφάσεις του Random Forest.
Αυτό βοηθά στην ερμηνεία του μοντέλου (σημαντικό σε εφαρμογές κυβερνοασφάλειας).


In [ ]:

# Υπολογισμός και ταξινόμηση feature importances
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=False)

# Εμφάνιση των 15 σημαντικότερων χαρακτηριστικών
top_n = 15
top_features = importances_sorted.head(top_n)

top_features


In [ ]:

# Οπτικοποίηση των 15 σημαντικότερων χαρακτηριστικών
ax = top_features.sort_values().plot(kind='barh')
ax.set_title(f'Top {len(top_features)} Feature Importances (Random Forest)')
ax.set_xlabel('Σημαντικότητα')
ax.set_ylabel('Χαρακτηριστικό')
plt.show()



## Συμπεράσματα

Το μοντέλο Random Forest μπορεί να διακρίνει αποτελεσματικά την Tor από τη Non-Tor κίνηση, με βάση τα χαρακτηριστικά δικτυακής κίνησης.
Επιπλέον, η ανάλυση feature importances προσφέρει ερμηνευσιμότητα, βοηθώντας να εντοπιστούν τα χαρακτηριστικά που επηρεάζουν περισσότερο την απόφαση ταξινόμησης.
